<a href="https://colab.research.google.com/github/sufianahmad513/FIRST/blob/main/email_frauddetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
import pandas as pd
import random

# Seed for reproducibility
random.seed(42)

# Templates for clean emails
clean_templates = [
    "Hi {name}, are we still on for our meeting at {time}? Let me know.",
    "Attached is the {topic} report for your review. Please send feedback.",
    "Hi team, just a reminder that the deadline for {topic} is tomorrow.",
    "Thanks for the quick response. I will follow up with {name} regarding the project.",
    "Can you send me the updated slides for the presentation on {topic}?",
    "Hi, please find the invoice for your recent subscription payment.",
    "Hey, let's grab lunch today at {time} near the office.",
    "Dear {name}, your appointment has been confirmed for tomorrow at {time}.",
    "The weekly sync has been rescheduled to {time}. See you there.",
    "Hi, I finished reviewing the draft you sent over. It looks great!"
]

# Templates for fraud/spam emails
fraud_templates = [
    "URGENT: Your bank account has been locked! Click here to verify your identity: {url}",
    "Congratulations! You have been selected to win a free {prize}! Claim now: {url}",
    "WARNING: Unusual login detected on your account. Reset your password immediately: {url}",
    "Get a guaranteed {prize} today! Just pay a small shipping fee at {url}",
    "Your subscription has expired. Update your billing information here to avoid disruption: {url}",
    "Earn up to $5,000/week working from home! Limited slots available, sign up now: {url}",
    "ALERT: Tax refund of {amount} is waiting for you. Fill out this form to claim: {url}",
    "You have received a secure document. Click here to sign in and view: {url}",
    "Double your investment in 24 hours! Safe and guaranteed crypto returns at {url}",
    "Dear customer, please update your account details immediately by visiting {url}"
]

names = ["John", "Sarah", "David", "Emma", "Michael", "Emily", "James", "Olivia"]
times = ["10:00 AM", "1:00 PM", "3:30 PM", "noon", "9:30 AM", "2:15 PM"]
topics = ["marketing", "budget", "design", "Q4 sales", "project development", "weekly sync"]
prizes = ["iPhone 15 Pro", "$1,000 Gift Card", "MacBook Air", "PlayStation 5"]
amounts = ["$450.00", "$1,250.99", "$89.50", "$2,100.00"]
urls = ["http://bit.ly/secure-verify-931", "http://login-update-auth.com", "http://free-rewards-center.net", "http://crypto-moon-profits.org", "http://verify-bank-id.info"]

emails = []

# Generate 100 Clean Emails
for _ in range(100):
    template = random.choice(clean_templates)
    text = template.format(
        name=random.choice(names),
        time=random.choice(times),
        topic=random.choice(topics)
    )
    emails.append((text, 0))

# Generate 100 Fraud Emails
for _ in range(100):
    template = random.choice(fraud_templates)
    text = template.format(
        prize=random.choice(prizes),
        amount=random.choice(amounts),
        url=random.choice(urls)
    )
    emails.append((text, 1))

# Shuffle and create DataFrame
random.shuffle(emails)
df_new = pd.DataFrame(emails, columns=['email_text', 'is_fraud'])

# Save to csv
df_new.to_csv('/content/mock_emails.csv', index=False)
print(f"Successfully generated and saved {len(df_new)} emails to '/content/mock_emails.csv'")
print("Class distribution:\n", df_new['is_fraud'].value_counts())


In [ ]:
print("--- Step 1: Loading Dataset ---")
# Using the generated mock dataset
try:
    df = pd.read_csv('/content/mock_emails.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Dataset 'mock_emails.csv' not found. Please create it first.")
    exit()

print(f"Total records: {len(df)}")
print(df.head())

In [ ]:
print("\n--- Step 2: Text Preprocessing & Vectorization ---")
# X: Email content, y: Label (1 = Fraud, 0 = Clean)
X = df['email_text']
y = df['is_fraud']

# Splitting dataset into training and testing configurations (80-20 Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transforming text data into numerical vectors using TF-IDF Vectorizer
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Training feature matrix shape: {X_train_vec.shape}")


In [ ]:
print("\n--- Step 3: Model Training (Naive Bayes) ---")
# Instantiating the classic text classification classifier
model = MultinomialNB()
model.fit(X_train_vec, y_train)
print("Model training completed.")

print("\n--- Step 4: Model Evaluation ---")
y_pred = model.predict(X_test_vec)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Clean Email', 'Fraud Email'], zero_division=0))

print("\n--- Step 5: Live Testing (Inference Pipeline) ---")
def test_email(text):
    text_vec = vectorizer.transform([text])
    prediction = model.predict(text_vec)[0]
    confidence = model.predict_proba(text_vec)[0]

    result = "🔴 FRAUD DETECTED" if prediction == 1 else "🟢 CLEAN EMAIL"
    prob = confidence[prediction] * 100
    print(f"\nEmail: \"{text}\"")
    print(f"Result: {result} ({prob:.2f}% Confidence)")

In [ ]:


# Test sentences
test_email("Hey John, can you send over the updated slides for the presentation?")
test_email("URGENT: Click here to claim your reward before the link expires!")


### Step 6: Exporting the Model and Vectorizer for Streamlit

In [ ]:
import pickle

# Save the model and the vectorizer to disk
with open('spam_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)

with open('vectorizer.pkl', 'wb') as vec_file:
    pickle.dump(vectorizer, vec_file)

print("Model and Vectorizer exported successfully!")

### Step 7: Creating the Streamlit App script (`app.py`)

In [ ]:
%%writefile app.py
import streamlit as st
import pickle

# Set page title and styling
st.set_page_config(page_title="Email Fraud & Spam Detector", page_icon="✉️", layout="centered")

st.title("✉️ Email Fraud & Spam Detector")
st.write("This application uses a Naive Bayes Classifier trained on email texts to detect potential fraud, phishing, or spam.")

# Load the saved model and vectorizer
@st.cache_resource
def load_artifacts():
    with open('spam_model.pkl', 'rb') as model_file:
        model = pickle.load(model_file)
    with open('vectorizer.pkl', 'rb') as vec_file:
        vectorizer = pickle.load(vec_file)
    return model, vectorizer

try:
    model, vectorizer = load_artifacts()
    st.success("Model artifacts loaded successfully!")
except Exception as e:
    st.error(f"Error loading model artifacts: {e}")

# Input area
user_input = st.text_area("Paste the email content below:", height=150, placeholder="Type or paste your email here...")

if st.button("Analyze Email", use_container_width=True):
    if user_input.strip() == "":
        st.warning("Please enter some email content first.")
    else:
        # Vectorize and predict
        text_vec = vectorizer.transform([user_input])
        prediction = model.predict(text_vec)[0]
        confidence = model.predict_proba(text_vec)[0]
        prob = confidence[prediction] * 100

        # Display result
        st.subheader("Analysis Result:")
        if prediction == 1:
            st.error(f"🔴 FRAUD DETECTED ({prob:.2f}% Confidence)")
            st.write("**Recommendation:** Do not click on any links, download attachments, or provide personal details.")
        else:
            st.success(f"🟢 CLEAN EMAIL ({prob:.2f}% Confidence)")
            st.write("**Recommendation:** This email looks safe, but always practice caution with unfamiliar senders.")

### How to Run Your App:

If you want to run this **locally** on your computer:
1. Download `app.py`, `spam_model.pkl`, and `vectorizer.pkl` from the Colab files panel on the left.
2. Install streamlit in your terminal: `pip install streamlit scikit-learn pandas`
3. Run the app: `streamlit run app.py`